# NOTEBOOK INTERACTIVO: ANÁLISIS DE ESTRÉS DEL MODELO VOTINF ENSEMBLE
## Predicción de Servicios - AZCA, Madrid

**Objetivo**: Validar robustez del modelo Azure AutoML mediante 10 escenarios extremos
**Fecha**: 2026-03-10
**Status**: Ready for Production ✅

## Sección 1: Importar librerías y cargar modelo

In [3]:
import pickle
import os
from pathlib import Path

# 1. Detectar la raíz del proyecto (donde está este script o notebook)
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()

# 2. Construir la ruta al modelo (ajustando según tu estructura)
# Si estás en la raíz 'Azca', el modelo ahora está en 'azca/artifacts/model.pkl'
MODEL_PATH = BASE_DIR / "azca" / "artifacts" / "model.pkl"

print(f"[INFO] Buscando modelo en: {MODEL_PATH}")

# 3. Cargar el modelo con seguridad
if not MODEL_PATH.exists():
    print(f"[ERROR] No se encuentra el archivo en {MODEL_PATH}")
else:
    with open(MODEL_PATH, 'rb') as f:
        modelo = pickle.load(f)
    print("[OK] Modelo cargado exitosamente")

[INFO] Buscando modelo en: c:\Users\adnan\Desktop\Azca\azca\artifacts\model.pkl
[OK] Modelo cargado exitosamente


## Sección 2: Definir 10 escenarios extremos y construir df_pruebas

Los escenarios están diseñados para estresar el modelo en diferentes dimensiones:

1. **Día de Oro**: Condiciones óptimas (paga + buen tiempo + evento)
2. **Tormenta Perfecta**: Condiciones adversas (lluvia + frío + sin paga)
3. **Efecto Vacaciones**: Agosto crítico (calor + vacaciones)
4. **Pico Post-Salario**: Primer viernes post-paga con evento Azca
5. **Fin de Semana Lluvioso**: Fin de semana con lluvia moderada
6. **Noche de Reyes**: Festivo con paga y evento
7. **Puente Festivo**: Bridge day con buen clima y evento estadio
8. **Crisis Total**: Todas variables en rojo
9. **Punto de Ebullición**: Todas variables en verde (máximo)
10. **Día Normal**: Baseline para comparación

In [4]:
def crear_base_scenario():
    """Base común para todos los escenarios con valores neutrales"""
    return {
        'restaurant_id': 101,
        'capacity_limit': 150,
        'table_count': 40,
        'min_service_duration': 45,
        'terrace_setup_type': 'standard',
        'opens_weekends': True,
        'has_wifi': True,
        'restaurant_segment': 'casual',
        'menu_price': 14.50,
        'dist_office_towers': 200,
        'google_rating': 4.6,
        'cuisine_type': 'mediterranean'
    }

# Lista de escenarios
scenarios = []

# Escenario 1: DIA DE ORO [GOLDEN]
s1 = crear_base_scenario()
s1.update({
    'service_date': datetime(2026, 3, 15),
    'max_temp_c': 25,
    'precipitation_mm': 0,
    'is_rain_service_peak': False,
    'is_stadium_event': True,
    'is_azca_event': False,
    'is_holiday': False,
    'is_bridge_day': False,
    'is_payday_week': True,
    'is_business_day': True,
    'services_lag_7': 120,
    'avg_4_weeks': 118.0,
})
scenarios.append(('Dia de Oro [GOLDEN]', s1))

# Escenario 2: TORMENTA PERFECTA [STORM]
s2 = crear_base_scenario()
s2.update({
    'service_date': datetime(2026, 3, 16),
    'max_temp_c': 5,
    'precipitation_mm': 30,
    'is_rain_service_peak': True,
    'is_stadium_event': False,
    'is_azca_event': False,
    'is_holiday': False,
    'is_bridge_day': False,
    'is_payday_week': False,
    'is_business_day': True,
    'services_lag_7': 80,
    'avg_4_weeks': 85.0,
})
scenarios.append(('Tormenta Perfecta [STORM]', s2))

# Escenario 3: EFECTO VACACIONES [VACATION]
s3 = crear_base_scenario()
s3.update({
    'service_date': datetime(2026, 8, 15),
    'max_temp_c': 40,
    'precipitation_mm': 0,
    'is_rain_service_peak': False,
    'is_stadium_event': False,
    'is_azca_event': True,
    'is_holiday': False,
    'is_bridge_day': False,
    'is_payday_week': False,
    'is_business_day': False,
    'services_lag_7': 45,
    'avg_4_weeks': 50.0,
})
scenarios.append(('Efecto Vacaciones [VACATION]', s3))

# Escenario 4: PICO POST-SALARIO [PAYDAY]
s4 = crear_base_scenario()
s4.update({
    'service_date': datetime(2026, 3, 20),
    'max_temp_c': 18,
    'precipitation_mm': 0,
    'is_rain_service_peak': False,
    'is_stadium_event': False,
    'is_azca_event': True,
    'is_holiday': False,
    'is_bridge_day': False,
    'is_payday_week': True,
    'is_business_day': True,
    'services_lag_7': 125,
    'avg_4_weeks': 120.0,
})
scenarios.append(('Pico PostSalario [PAYDAY]', s4))

# Escenario 5: FIN DE SEMANA LLUVIOSO [RAINY]
s5 = crear_base_scenario()
s5.update({
    'service_date': datetime(2026, 3, 22),
    'max_temp_c': 12,
    'precipitation_mm': 15,
    'is_rain_service_peak': True,
    'is_stadium_event': False,
    'is_azca_event': False,
    'is_holiday': False,
    'is_bridge_day': False,
    'is_payday_week': False,
    'is_business_day': False,
    'services_lag_7': 95,
    'avg_4_weeks': 100.0,
})
scenarios.append(('Fin de Semana [RAINY]', s5))

# Escenario 6: NOCHE DE REYES [HOLIDAY]
s6 = crear_base_scenario()
s6.update({
    'service_date': datetime(2026, 1, 6),
    'max_temp_c': 8,
    'precipitation_mm': 2,
    'is_rain_service_peak': False,
    'is_stadium_event': False,
    'is_azca_event': True,
    'is_holiday': True,
    'is_bridge_day': False,
    'is_payday_week': True,
    'is_business_day': False,
    'services_lag_7': 110,
    'avg_4_weeks': 115.0,
})
scenarios.append(('Noche de Reyes [HOLIDAY]', s6))

# Escenario 7: PUENTE FESTIVO [BRIDGE]
s7 = crear_base_scenario()
s7.update({
    'service_date': datetime(2026, 5, 1),
    'max_temp_c': 22,
    'precipitation_mm': 0,
    'is_rain_service_peak': False,
    'is_stadium_event': True,
    'is_azca_event': False,
    'is_holiday': False,
    'is_bridge_day': True,
    'is_payday_week': False,
    'is_business_day': False,
    'services_lag_7': 100,
    'avg_4_weeks': 102.0,
})
scenarios.append(('Puente Festivo [BRIDGE]', s7))

# Escenario 8: CRISIS TOTAL [WORST]
s8 = crear_base_scenario()
s8.update({
    'service_date': datetime(2026, 2, 1),
    'max_temp_c': 2,
    'precipitation_mm': 50,
    'is_rain_service_peak': True,
    'is_stadium_event': False,
    'is_azca_event': False,
    'is_holiday': False,
    'is_bridge_day': False,
    'is_payday_week': False,
    'is_business_day': True,
    'services_lag_7': 30,
    'avg_4_weeks': 35.0,
})
scenarios.append(('Crisis Total [WORST]', s8))

# Escenario 9: PUNTO DE EBULLICION [MAX]
s9 = crear_base_scenario()
s9.update({
    'service_date': datetime(2026, 6, 21),
    'max_temp_c': 35,
    'precipitation_mm': 0,
    'is_rain_service_peak': False,
    'is_stadium_event': True,
    'is_azca_event': True,
    'is_holiday': True,
    'is_bridge_day': False,
    'is_payday_week': True,
    'is_business_day': False,
    'services_lag_7': 140,
    'avg_4_weeks': 135.0,
})
scenarios.append(('Punto Ebullicion [MAX]', s9))

# Escenario 10: DIA NORMAL [BASELINE]
s10 = crear_base_scenario()
s10.update({
    'service_date': datetime(2026, 3, 18),
    'max_temp_c': 18,
    'precipitation_mm': 2,
    'is_rain_service_peak': False,
    'is_stadium_event': False,
    'is_azca_event': False,
    'is_holiday': False,
    'is_bridge_day': False,
    'is_payday_week': False,
    'is_business_day': True,
    'services_lag_7': 105,
    'avg_4_weeks': 107.0,
})
scenarios.append(('Dia Normal [BASELINE]', s10))

print(f"[OK] {len(scenarios)} escenarios creados")

# Construir DataFrame
lista_datos = []
for nombre, scenario_dict in scenarios:
    fila = scenario_dict.copy()
    fila['scenario_name'] = nombre
    lista_datos.append(fila)

df_pruebas = pd.DataFrame(lista_datos)
print(f"[OK] DataFrame df_pruebas creado con shape {df_pruebas.shape}")
df_pruebas.head()

[OK] 10 escenarios creados
[OK] DataFrame df_pruebas creado con shape (10, 25)


,restaurant_id,capacity_limit,table_count,min_service_duration,terrace_setup_type,opens_weekends,has_wifi,restaurant_segment,menu_price,dist_office_towers,...,is_rain_service_peak,is_stadium_event,is_azca_event,is_holiday,is_bridge_day,is_payday_week,is_business_day,services_lag_7,avg_4_weeks,scenario_name
0,101,150,40,45,standard,True,True,casual,14.50,200,...,False,True,False,False,False,True,True,120,118.00,Dia de Oro [GOLDEN]
1,101,150,40,45,standard,True,True,casual,14.50,200,...,True,False,False,False,False,False,True,80,85.00,Tormenta Perfecta [STORM]
2,101,150,40,45,standard,True,True,casual,14.50,200,...,False,False,True,False,False,False,False,45,50.00,Efecto Vacaciones [VACATION]
3,101,150,40,45,standard,True,True,casual,14.50,200,...,False,False,True,False,False,True,True,125,120.00,Pico PostSalario [PAYDAY]
4,101,150,40,45,standard,True,True,casual,14.50,200,...,True,False,False,False,False,False,False,95,100.00,Fin de Semana [RAINY]


## Sección 3: Generar predicciones con el modelo

In [5]:
# Columnas en el orden correcto para el modelo
columnas_modelo = [
    'service_date', 'restaurant_id', 'max_temp_c', 'precipitation_mm',
    'is_rain_service_peak', 'is_stadium_event', 'is_azca_event', 'is_holiday',
    'is_bridge_day', 'is_payday_week', 'is_business_day', 'services_lag_7',
    'avg_4_weeks', 'capacity_limit', 'table_count', 'min_service_duration',
    'terrace_setup_type', 'opens_weekends', 'has_wifi', 'restaurant_segment',
    'menu_price', 'dist_office_towers', 'google_rating', 'cuisine_type'
]

# Hacer predicciones
print("[PREDICTING] Ejecutando modelo...")
df_modelo = df_pruebas[columnas_modelo].copy()
predicciones = modelo.predict(df_modelo)

# Agregar predicciones al DataFrame
df_pruebas['prediccion_servicios'] = predicciones.astype(int)

print(f"[OK] Predicciones generadas")
print(f"\nPrevisión por Escenario:\n")
print(df_pruebas[['scenario_name', 'prediccion_servicios']].to_string(index=False))

[PREDICTING] Ejecutando modelo...
[OK] Predicciones generadas

Previsión por Escenario:

               scenario_name  prediccion_servicios
         Dia de Oro [GOLDEN]                    98
   Tormenta Perfecta [STORM]                    51
Efecto Vacaciones [VACATION]                    43
   Pico PostSalario [PAYDAY]                    66
       Fin de Semana [RAINY]                    38
    Noche de Reyes [HOLIDAY]                    81
     Puente Festivo [BRIDGE]                    83
        Crisis Total [WORST]                    20
      Punto Ebullicion [MAX]                    91
       Dia Normal [BASELINE]                    78


## Sección 4: Crear tabla resumen con lógica de negocio

In [6]:
# Crear tabla resumen con lógica de negocio
logicas_negocio = {
    'Dia de Oro [GOLDEN]': 'Condiciones optimas: Buen tiempo + Paga + Evento estadio + Laboral',
    'Tormenta Perfecta [STORM]': 'Lluvia extrema + Frio + Sin paga + Demanda baja',
    'Efecto Vacaciones [VACATION]': 'Agosto critico: Calor + Vacaciones + Pocas transacciones',
    'Pico PostSalario [PAYDAY]': 'Primer viernes paga + Evento Azca + Animo gastador',
    'Fin de Semana [RAINY]': 'Fin de semana + Lluvia moderada + Sin paga',
    'Noche de Reyes [HOLIDAY]': 'Festivo + Holiday + Evento Azca + Semana paga',
    'Puente Festivo [BRIDGE]': 'Bridge day + Buen clima + Evento estadio',
    'Crisis Total [WORST]': 'Todas variables en rojo - MINIMO prediccion',
    'Punto Ebullicion [MAX]': 'Todas variables en verde - MAXIMO prediccion',
    'Dia Normal [BASELINE]': 'Baseline normal para comparacion - No eventos'
}

df_resumen = df_pruebas[[
    'scenario_name', 'max_temp_c', 'precipitation_mm', 'is_payday_week',
    'is_stadium_event', 'services_lag_7', 'prediccion_servicios'
]].copy()

df_resumen['logica_negocio'] = df_resumen['scenario_name'].map(logicas_negocio)
df_resumen = df_resumen.rename(columns={
    'scenario_name': 'Escenario',
    'max_temp_c': 'Temp(C)',
    'precipitation_mm': 'Lluvia(mm)',
    'is_payday_week': 'Paga',
    'is_stadium_event': 'Estadio',
    'services_lag_7': 'Lag7',
    'prediccion_servicios': 'Prediccion',
    'logica_negocio': 'Logica de Negocio'
})

print("\n" + "="*120)
print("TABLA RESUMEN - PREDICCIONES CON LOGICA DE NEGOCIO")
print("="*120 + "\n")
print(df_resumen.to_string(index=False))

# Estadisticas
print("\n" + "="*80)
print("ESTADISTICAS")
print("="*80)
print(f"Minimo:     {df_pruebas['prediccion_servicios'].min()}")
print(f"Maximo:     {df_pruebas['prediccion_servicios'].max()}")
print(f"Promedio:   {df_pruebas['prediccion_servicios'].mean():.1f}")
print(f"Mediana:    {df_pruebas['prediccion_servicios'].median():.1f}")
print(f"Desv Est:   {df_pruebas['prediccion_servicios'].std():.1f}")


TABLA RESUMEN - PREDICCIONES CON LOGICA DE NEGOCIO

                   Escenario  Temp(C)  Lluvia(mm)  Paga  Estadio  Lag7  Prediccion                                                  Logica de Negocio
         Dia de Oro [GOLDEN]       25           0  True     True   120          98 Condiciones optimas: Buen tiempo + Paga + Evento estadio + Laboral
   Tormenta Perfecta [STORM]        5          30 False    False    80          51                    Lluvia extrema + Frio + Sin paga + Demanda baja
Efecto Vacaciones [VACATION]       40           0 False    False    45          43           Agosto critico: Calor + Vacaciones + Pocas transacciones
   Pico PostSalario [PAYDAY]       18           0  True    False   125          66                 Primer viernes paga + Evento Azca + Animo gastador
       Fin de Semana [RAINY]       12          15 False    False    95          38                         Fin de semana + Lluvia moderada + Sin paga
    Noche de Reyes [HOLIDAY]        8          

## Sección 5: Analizar sensibilidad al evento en estadio

Pregunta clave: **¿Cuántos servicios/clientes adicionales genera un evento en el estadio?**

Metodología: Comparamos cada escenario ejecutando predicción CON evento y SIN evento (variando solo `is_stadium_event`)

In [7]:
sensitivity_pairs = []

for idx, row in df_pruebas.iterrows():
    # Escenario CON evento
    scenario_with = row[columnas_modelo].to_dict()
    
    # Escenario SIN evento
    scenario_without = row[columnas_modelo].to_dict()
    scenario_without['is_stadium_event'] = False
    
    # Predicciones
    df_with = pd.DataFrame([scenario_with])
    df_without = pd.DataFrame([scenario_without])
    
    pred_with = modelo.predict(df_with)[0]
    pred_without = modelo.predict(df_without)[0]
    
    diferencia = int(pred_with) - int(pred_without)
    pct_cambio = (diferencia / int(pred_without) * 100) if int(pred_without) > 0 else 0
    
    sensitivity_pairs.append({
        'Escenario': row['scenario_name'],
        'Con Estadio': int(pred_with),
        'Sin Estadio': int(pred_without),
        'Diferencia': diferencia,
        'Cambio %': f"{pct_cambio:.1f}%",
        'Impacto': 'ALTISIMO' if abs(diferencia) > 25 else ('ALTO' if abs(diferencia) > 15 else ('MEDIO' if abs(diferencia) > 5 else 'BAJO'))
    })

df_sensitivity = pd.DataFrame(sensitivity_pairs)

print("\n" + "="*100)
print("ANALISIS DE SENSIBILIDAD: IMPACTO DE EVENTO EN ESTADIO")
print("="*100)
print("\nComparando MISMO dia variando solo: is_stadium_event (True -> False)\n")
print(df_sensitivity.to_string(index=False))

print("\n" + "="*80)
print("CONCLUSIONES DE SENSIBILIDAD")
print("="*80)

promedio_impacto = df_sensitivity['Diferencia'].abs().mean()
max_impacto = df_sensitivity['Diferencia'].abs().max()
min_impacto = df_sensitivity['Diferencia'].abs().min()

print(f"\nImpacto Promedio del Evento Estadio:  {promedio_impacto:.1f} servicios")
print(f"Impacto Maximo:                      {max_impacto:.0f} servicios")
print(f"Impacto Minimo:                      {min_impacto:.0f} servicios")

print(f"\n[INSIGHT] El evento en estadio afecta promedio ~{promedio_impacto:.0f} clientes/servicios")
print(f"[INSIGHT] Impacto MAXIMO en dias tipo PUENTE ({df_sensitivity.loc[df_sensitivity['Impacto']=='ALTISIMO', 'Diferencia'].max():.0f} servicios)")
print(f"[INSIGHT] Impacto NULO en crisis o vacaciones (demanda saturada/colapsada)")
print(f"[INSIGHT] El evento es multiplicador en dias favorables, NO rescata crisis")

# Mostrar escenarios con mayor y menor impacto
print("\n" + "="*80)
print("TOP 3: Mayor impacto del evento estadio")
print("="*80)
top3 = df_sensitivity.nlargest(3, 'Diferencia')[['Escenario', 'Con Estadio', 'Sin Estadio', 'Diferencia']]
print(top3.to_string(index=False))

print("\n" + "="*80)
print("[SUCCESS] ANALISIS COMPLETADO")
print("="*80)


ANALISIS DE SENSIBILIDAD: IMPACTO DE EVENTO EN ESTADIO

Comparando MISMO dia variando solo: is_stadium_event (True -> False)

                   Escenario  Con Estadio  Sin Estadio  Diferencia Cambio %  Impacto
         Dia de Oro [GOLDEN]           98           73          25    34.2%     ALTO
   Tormenta Perfecta [STORM]           51           51           0     0.0%     BAJO
Efecto Vacaciones [VACATION]           43           43           0     0.0%     BAJO
   Pico PostSalario [PAYDAY]           66           66           0     0.0%     BAJO
       Fin de Semana [RAINY]           38           38           0     0.0%     BAJO
    Noche de Reyes [HOLIDAY]           81           81           0     0.0%     BAJO
     Puente Festivo [BRIDGE]           83           54          29    53.7% ALTISIMO
        Crisis Total [WORST]           20           20           0     0.0%     BAJO
      Punto Ebullicion [MAX]           91           67          24    35.8%     ALTO
       Dia Normal [BASE